# 09 - Detector ONNX parity re-check (criterion A9b)

Criterion A9b: maximum absolute difference between the PyTorch detector and its ONNX export, **≤ 0.001**.
The original check (seed 0, 4 random 640×640 inputs, CPU, `deploy.yolo_parity`) recorded 0.0018, and a later re-check 0.0020.

This notebook tries several export and runtime settings on the **same weights** and the **same protocol**, and records each result.
The limit is not changed. A setting counts as passing only if its maximum difference on the original protocol is ≤ 0.001
**and** it is used unchanged by the service afterwards.

Run time: about 20–30 minutes on a CPU runtime (no GPU needed); step 4 downloads Oxford-IIIT Pet (10–20 minutes).


In [ ]:
# 1. Setup: code from GitHub, pinned versions
!git clone -q https://github.com/Yasser-20-alhajlah/smart-pet-vision /content/spv-repo 2>/dev/null || git -C /content/spv-repo pull -q
!pip -q install ultralytics==8.4.182 onnx==1.23.2 onnxruntime==1.31.0 onnxslim
import sys, os, json, hashlib, platform
sys.path.insert(0, "/content/spv-repo/smart-pet-vision/phase2/src")
import numpy as np, torch, onnx, onnxruntime as ort, ultralytics
print("torch", torch.__version__, "| onnx", onnx.__version__, "| onnxruntime", ort.__version__, "| ultralytics", ultralytics.__version__)


In [ ]:
# 2. Find the detector weights on Drive and check they are the evaluated ones
from google.colab import drive
drive.mount("/content/drive")
EXPECTED_SHA256 = "f48e4d8ea5c925c18a4b43744c2f697bfc01a0a473492275f000848125da1cd4"  # detector best.pt (team record)
import glob
cands = glob.glob("/content/drive/MyDrive/**/best.pt", recursive=True)
def sha(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
WEIGHTS = next((p for p in cands if sha(p) == EXPECTED_SHA256), None)
assert WEIGHTS, f"Detector best.pt with the expected SHA-256 not found among {len(cands)} best.pt files. Set WEIGHTS manually."
print("Detector weights:", WEIGHTS)
OUT = "/content/onnx_recheck"; os.makedirs(OUT, exist_ok=True)
IMGSZ = 640


In [ ]:
# 3. Export variants and measure them on the ORIGINAL protocol (seed 0, 4 random inputs, CPU)
import shutil, tempfile
from ultralytics import YOLO

def export(name, **kw):
    dst = f"{OUT}/{name}.onnx"
    with tempfile.TemporaryDirectory() as d:
        src = os.path.join(d, "best.pt"); shutil.copy2(WEIGHTS, src)   # never touch the original file
        p = YOLO(src).export(format="onnx", imgsz=IMGSZ, dynamic=False, **kw)
        shutil.copy2(p, dst)
    return dst

def parity(onnx_path, opt_level="all", threads=0, n=4, seed=0):
    # Same steps as spv.deploy.yolo_parity; only the ONNX Runtime session options can differ.
    torch.manual_seed(seed)
    net = YOLO(WEIGHTS).model.float().cpu().eval()
    so = ort.SessionOptions()
    so.graph_optimization_level = {"all": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
                                   "basic": ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
                                   "off": ort.GraphOptimizationLevel.ORT_DISABLE_ALL}[opt_level]
    if threads: so.intra_op_num_threads = threads
    sess = ort.InferenceSession(onnx_path, so, providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    diffs = []
    for _ in range(n):
        x = torch.rand(1, 3, IMGSZ, IMGSZ)
        with torch.no_grad():
            ref = net(x)
        ref = (ref[0] if isinstance(ref, (list, tuple)) else ref).numpy()
        out = sess.run(None, {name: x.numpy()})[0]
        assert out.shape == ref.shape
        diffs.append(float(np.abs(out - ref).max()))
    return max(diffs)

variants = [
    ("A_original_opset17",            dict(opset=17, simplify=False), "all", 0),
    ("B_opset17_ort_opt_off",         dict(opset=17, simplify=False), "off", 0),
    ("C_opset17_ort_basic",           dict(opset=17, simplify=False), "basic", 0),
    ("D_opset17_ort_opt_off_1thread", dict(opset=17, simplify=False), "off", 1),
    ("E_opset12_simplified",          dict(opset=12, simplify=True),  "all", 0),
    ("F_opset12_simplified_opt_off",  dict(opset=12, simplify=True),  "off", 0),
]
results, exported = [], {}
for name, kw, opt, thr in variants:
    key = json.dumps(kw, sort_keys=True)
    try:
        if key not in exported:
            exported[key] = export(name, **kw)
        d = parity(exported[key], opt, thr)
    except Exception as e:
        results.append({"variant": name, "export": kw, "error": str(e)[:300], "passed_A9b": False})
        print(f"{name:32s} failed: {e}"); continue
    results.append({"variant": name, "export": kw, "ort_graph_optimization": opt, "ort_threads": thr or "default",
                    "max_abs_diff": d, "tolerance": 1e-3, "passed_A9b": d <= 1e-3})
    print(f"{name:32s} max_abs_diff={d:.6f}  {'PASS' if d <= 1e-3 else 'not met'}")


In [ ]:
# 3b. How precise is the PyTorch reference itself? Compare float32 PyTorch with float64 PyTorch (near-exact)
net32 = YOLO(WEIGHTS).model.float().cpu().eval()
net64 = YOLO(WEIGHTS).model.double().cpu().eval()
torch.manual_seed(0)
xs = [torch.rand(1, 3, IMGSZ, IMGSZ) for _ in range(4)]
first = lambda r: r[0] if isinstance(r, (list, tuple)) else r
with torch.no_grad():
    r32 = [first(net32(x)).numpy() for x in xs]
    r64 = [first(net64(x.double())).numpy() for x in xs]
sess = ort.InferenceSession(exported[json.dumps(dict(opset=17, simplify=False), sort_keys=True)], providers=["CPUExecutionProvider"])
o32 = [sess.run(None, {sess.get_inputs()[0].name: x.numpy()})[0] for x in xs]
precision = {"pytorch_fp32_vs_fp64": max(float(np.abs(a - b).max()) for a, b in zip(r32, r64)),
             "onnx_fp32_vs_fp64": max(float(np.abs(a - b).max()) for a, b in zip(o32, r64))}
print(precision)
print("If pytorch_fp32_vs_fp64 is above 0.001, the float32 reference itself is not accurate to 0.001.")


In [ ]:
# 4. Real-image check of the best variant: final detections on the first 50 Oxford test images
# Downloads Oxford-IIIT Pet (10-20 minutes). Without runs/data/metadata.csv the split is rebuilt with the project's seed 42.
best = min((r for r in results if "max_abs_diff" in r), key=lambda r: r["max_abs_diff"])
print("Best variant:", best["variant"], best["max_abs_diff"])
try:
    from spv.common import load_config
    from spv import data, deploy
    os.chdir("/content/spv-repo/smart-pet-vision/phase2")
    cfg = load_config()
    df = data.ensure_data(cfg)
    imgs = [data.dataset_dir(cfg) / "images" / f"{n}.jpg" for n in df[df.split == "test"].image.head(50)]
    onnx_path = exported[json.dumps(best["export"], sort_keys=True)]
    real = deploy.yolo_parity_real(WEIGHTS, onnx_path, imgs, IMGSZ, conf=0.40, iou=0.50)
    real["note"] = "deploy.yolo_parity_real uses default ONNX Runtime options"
    best["real_images_50"] = real
    print(real)
except Exception as e:
    print("Real-image check skipped:", e)


In [ ]:
# 5. Save the record (copy this file back to the team folder)
record = {"criterion": "A9b detector ONNX max abs diff <= 0.001 (unchanged)",
          "protocol": "seed 0, 4 random 640x640 inputs, CPU, same steps as spv.deploy.yolo_parity",
          "weights": WEIGHTS, "weights_sha256": EXPECTED_SHA256,
          "versions": {"torch": torch.__version__, "onnx": onnx.__version__, "onnxruntime": ort.__version__,
                       "ultralytics": ultralytics.__version__, "python": platform.python_version()},
          "results": results, "float32_precision": precision,
          "any_variant_passed": any(r["passed_A9b"] for r in results)}
path = f"{OUT}/onnx_parity_recheck.json"
json.dump(record, open(path, "w"), indent=2)
dst = "/content/drive/MyDrive/onnx_parity_recheck_A9b.json"
shutil.copy2(path, dst); print("Saved:", dst)
print(json.dumps({r['variant']: round(r['max_abs_diff'], 6) if 'max_abs_diff' in r else r['error'] for r in results}, indent=1))


## How to read the result

- **A variant passes** (max_abs_diff ≤ 0.001): A9b can be reported as met **for that export and runtime setting**, provided the service is configured the same way (for example `graph_optimization_level = ORT_DISABLE_ALL` in the ONNX session) and the latency benchmark is repeated with it.
- **No variant passes:** A9b stays *not met*, and the report keeps the no-go for the ONNX detector.
- **Cell 3b** shows whether the limit is below float32 precision: on the public YOLO11n model (tested before this notebook was shared) float32 PyTorch was 0.00137 from the float64 result and the float32 ONNX export 0.0021, so no float32 export could reliably reach 0.001. A float64 ONNX export is not an option on CPU, because ONNX Runtime has no float64 convolution there.
